# Week 5 – Apache Spark: Data Cleaning, Transformation & Aggregation
**Internship Assignment | Data Engineering Track | Celebal Technologies**

---

### Objective
Learn the basics of Apache Spark and use it to clean, transform, and analyze data using DataFrames.

### Dataset
- **File:** `ecommerce_data.csv`
- **Rows:** ~1,030 (including ~30 injected duplicates)
- **Columns:** transaction_id, user_id, age, product_category, region, city, price, status, transaction_date, subscription


---
## Step 1 – Understanding the Basics

### What is MapReduce?
MapReduce is a programming model for processing large datasets across a cluster. It works in two phases:
- **Map:** each node reads data from disk, applies a function, and writes intermediate output back to disk.
- **Reduce:** reads those intermediate files from disk again, aggregates, and writes final output.

**Problem:** Every phase involves disk I/O. For iterative algorithms (ML, graph processing), this means dozens of full disk reads/writes.

### Why Spark is Better

| Feature | MapReduce | Apache Spark |
|---------|-----------|--------------|
| Data storage | Disk (HDFS) after every step | RAM (in-memory) |
| Speed | Slow for iteration | Up to 100× faster |
| API | Low-level (Java) | High-level DataFrame / SQL |
| Use cases | Batch only | Batch, Streaming, ML, SQL, Graph |
| Fault tolerance | Replication | DAG recomputation (lineage) |

**Key Spark concepts:**
- **RDD (Resilient Distributed Dataset):** low-level immutable distributed collection
- **DataFrame:** table-like structure with named columns (built on top of RDD)
- **Lazy evaluation:** transformations are not executed until an action (`.show()`, `.count()`, `.write`) is triggered
- **DAG (Directed Acyclic Graph):** Spark's execution plan — Catalyst optimizer reorders/combines operations for efficiency


---
## Step 2 – Start Spark: Create a Spark Session


In [1]:
# Import all required Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, avg, sum as spark_sum, count, min as spark_min, max as spark_max,
    round as spark_round, trim, to_date, when, isnull, regexp_replace,
    coalesce, stddev
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType
)

# Create Spark Session — entry point for all Spark operations
spark = (
    SparkSession.builder
    .appName("Week5_Spark_EcommerceAnalysis")
    .master("local[*]")                        # use all available CPU cores locally
    .config("spark.sql.shuffle.partitions", "4")  # reduce shuffle partitions for local mode
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

# Suppress verbose logs — only show WARN and above
spark.sparkContext.setLogLevel("WARN")

print("✅ Spark Session created successfully!")
print(f"   App Name : {spark.sparkContext.appName}")
print(f"   Version  : {spark.version}")
print(f"   Master   : {spark.sparkContext.master}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/28 18:28:03 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/06/28 18:28:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/28 18:28:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Session created successfully!
   App Name : Week5_Spark_EcommerceAnalysis
   Version  : 4.1.2
   Master   : local[*]


---
## Step 3 – Load Data into a Spark DataFrame


In [2]:
# Define the schema explicitly — safer than inferSchema=True
# inferSchema causes an extra full scan and can misidentify date formats
schema = StructType([
    StructField("transaction_id",   IntegerType(),  True),
    StructField("user_id",          StringType(),   True),
    StructField("age",              IntegerType(),  True),
    StructField("product_category", StringType(),   True),
    StructField("region",           StringType(),   True),
    StructField("city",             StringType(),   True),
    StructField("price",            DoubleType(),   True),
    StructField("status",           StringType(),   True),
    StructField("transaction_date", StringType(),   True),  # read as string; cast later
    StructField("subscription",     StringType(),   True),
])

# Load CSV into DataFrame
df_raw = (
    spark.read
    .option("header", "true")
    .option("nullValue", "")        # treat empty strings as null during load
    .schema(schema)
    .csv("data/ecommerce_data.csv")
)

print(f"✅ Data loaded successfully!")
print(f"   Total rows : {df_raw.count()}")
print(f"   Columns    : {len(df_raw.columns)}")


✅ Data loaded successfully!


   Total rows : 1030
   Columns    : 10


In [3]:
# View first 5 rows
print("=== First 5 Rows ===")
df_raw.show(5, truncate=False)


=== First 5 Rows ===


+--------------+---------+---+----------------+-------+-------+-------+-------+----------------+------------+
|transaction_id|user_id  |age|product_category|region |city   |price  |status |transaction_date|subscription|
+--------------+---------+---+----------------+-------+-------+-------+-------+----------------+------------+
|529           |user_0529|24 |Electronics     |South  |Jaipur |1087.34|NULL   |2024-09-22      |Premium     |
|847           |user_0847|19 |Books           |Central|Chennai|1378.9 |NULL   |2024-12-13      |Premium     |
|431           |user_0431|34 |Books           |East   |Mumbai |4725.49|Pending|2024-07-16      |Basic       |
|602           |user_0602|32 |Clothing        |South  |Lucknow|3834.8 |Pending|2024-08-28      |Standard    |
|421           |user_0421|34 |Clothing        |Central|Mumbai |3456.07|Pending|2024-05-31      |Standard    |
+--------------+---------+---+----------------+-------+-------+-------+-------+----------------+------------+
only showi

In [4]:
# View column names and data types (schema)
print("=== Schema ===")
df_raw.printSchema()


=== Schema ===
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- subscription: string (nullable = true)



In [5]:
# Quick summary statistics for numeric columns
print("=== Summary Statistics ===")
df_raw.describe("age", "price").show()


=== Summary Statistics ===


+-------+-----------------+------------------+
|summary|              age|             price|
+-------+-----------------+------------------+
|  count|             1030|               992|
|   mean|39.12038834951456|2586.5313205645157|
| stddev|12.32563802075647|1453.2317526197846|
|    min|               18|             12.71|
|    max|               60|           4999.54|
+-------+-----------------+------------------+



In [6]:
# Count nulls per column — baseline quality check
from pyspark.sql.functions import isnan

print("=== Null Count per Column ===")
null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
null_counts.show()


=== Null Count per Column ===


+--------------+-------+---+----------------+------+----+-----+------+----------------+------------+
|transaction_id|user_id|age|product_category|region|city|price|status|transaction_date|subscription|
+--------------+-------+---+----------------+------+----+-----+------+----------------+------------+
|             0|      0|  0|               0|     0|   0|   38|   278|               0|           0|
+--------------+-------+---+----------------+------+----+-----+------+----------------+------------+



---
## Step 4 – Data Cleaning

Data cleaning is always the first real step in any pipeline. We handle:
1. Duplicate rows
2. Null / missing values
3. Inconsistent data (whitespace, wrong types)


In [7]:
# ── 4a. Remove Duplicate Rows ──────────────────────────────────────────────

before = df_raw.count()

# dropDuplicates() with no args: removes rows where ALL columns are identical
df_deduped = df_raw.dropDuplicates()

after = df_deduped.count()
print(f"Rows before deduplication : {before}")
print(f"Rows after  deduplication : {after}")
print(f"Duplicates removed        : {before - after}")


Rows before deduplication : 1030
Rows after  deduplication : 1000
Duplicates removed        : 30


In [8]:
# ── 4b. Handle Missing Values ──────────────────────────────────────────────

# Strategy:
#   - price  → fill with 0 (null price = no charge recorded, treat as 0)
#   - status → fill with 'Unknown'
#   - age    → drop rows where age is null (cannot impute meaningfully)

# Step 1: Fill nulls for price and status
df_filled = df_deduped.na.fill({
    "price":  0.0,
    "status": "Unknown"
})

# Step 2: Drop remaining rows where age is null
df_no_null = df_filled.na.drop(subset=["age"])

print(f"Rows after fill+drop: {df_no_null.count()}")

# Verify nulls are gone
print("\n=== Null check after cleaning ===")
df_no_null.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in ["age", "price", "status"]
]).show()


Rows after fill+drop: 1000

=== Null check after cleaning ===


+---+-----+------+
|age|price|status|
+---+-----+------+
|  0|    0|     0|
+---+-----+------+



In [9]:
# ── 4c. Fix Inconsistent Data ──────────────────────────────────────────────

# Problem 1: trim whitespace from string columns (common in CSV exports)
# Problem 2: cast transaction_date from String → DateType
# Problem 3: ensure age is within valid range (18–100)

df_clean = (
    df_no_null
    # Trim whitespace from all string columns
    .withColumn("user_id",          trim(col("user_id")))
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("region",           trim(col("region")))
    .withColumn("city",             trim(col("city")))
    .withColumn("status",           trim(col("status")))
    .withColumn("subscription",     trim(col("subscription")))
    # Cast transaction_date string → DateType
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd"))
    # Flag rows with out-of-range age as null, then drop
    .withColumn("age", when((col("age") >= 18) & (col("age") <= 100), col("age")).otherwise(None))
    .na.drop(subset=["age"])
)

print(f"✅ Cleaned DataFrame: {df_clean.count()} rows")
print("\n=== Schema after cleaning ===")
df_clean.printSchema()
df_clean.show(5, truncate=False)


✅ Cleaned DataFrame: 1000 rows

=== Schema after cleaning ===
root
 |-- transaction_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- product_category: string (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- price: double (nullable = false)
 |-- status: string (nullable = false)
 |-- transaction_date: date (nullable = true)
 |-- subscription: string (nullable = true)



+--------------+---------+---+----------------+-------+---------+-------+--------+----------------+------------+
|transaction_id|user_id  |age|product_category|region |city     |price  |status  |transaction_date|subscription|
+--------------+---------+---+----------------+-------+---------+-------+--------+----------------+------------+
|463           |user_0463|29 |Sports          |Central|Lucknow  |756.31 |Active  |2024-02-19      |Basic       |
|920           |user_0920|30 |Clothing        |South  |Mumbai   |1672.84|Active  |2024-08-28      |Basic       |
|683           |user_0683|28 |Books           |East   |Bangalore|4781.51|Inactive|2024-11-12      |Basic       |
|191           |user_0191|46 |Sports          |West   |Ahmedabad|3446.18|Inactive|2024-03-21      |Standard    |
|861           |user_0861|53 |Clothing        |North  |Kolkata  |985.08 |Inactive|2024-04-12      |Basic       |
+--------------+---------+---+----------------+-------+---------+-------+--------+--------------

---
## Step 5 – Filter Data

Filtering applies **row-level** conditions. This is a **narrow transformation** — each partition is processed independently with no data movement across the network.


In [10]:
# ── 5a. Filter by Age (18–30, inclusive) ──────────────────────────────────
df_young = df_clean.filter(col("age").between(18, 30))

print(f"Users aged 18–30: {df_young.count()} rows")
df_young.show(5)


Users aged 18–30: 310 rows


+--------------+---------+---+----------------+-------+---------+-------+--------+----------------+------------+
|transaction_id|  user_id|age|product_category| region|     city|  price|  status|transaction_date|subscription|
+--------------+---------+---+----------------+-------+---------+-------+--------+----------------+------------+
|           463|user_0463| 29|          Sports|Central|  Lucknow| 756.31|  Active|      2024-02-19|       Basic|
|           920|user_0920| 30|        Clothing|  South|   Mumbai|1672.84|  Active|      2024-08-28|       Basic|
|           683|user_0683| 28|           Books|   East|Bangalore|4781.51|Inactive|      2024-11-12|       Basic|
|           646|user_0646| 19|     Electronics|   East|   Mumbai|2674.36| Pending|      2024-08-18|       Basic|
|           774|user_0774| 26|        Clothing|  North|  Chennai|2270.98| Pending|      2024-10-18|     Premium|
+--------------+---------+---+----------------+-------+---------+-------+--------+--------------

In [11]:
# ── 5b. Filter by Category ────────────────────────────────────────────────
df_electronics = df_clean.filter(col("product_category") == "Electronics")

print(f"Electronics transactions: {df_electronics.count()} rows")
df_electronics.show(5)


Electronics transactions: 182 rows


+--------------+---------+---+----------------+-------+-------+-------+--------+----------------+------------+
|transaction_id|  user_id|age|product_category| region|   city|  price|  status|transaction_date|subscription|
+--------------+---------+---+----------------+-------+-------+-------+--------+----------------+------------+
|           646|user_0646| 19|     Electronics|   East| Mumbai|2674.36| Pending|      2024-08-18|       Basic|
|           984|user_0984| 40|     Electronics|  North|Kolkata|1649.47|Inactive|      2024-06-30|    Standard|
|           306|user_0306| 48|     Electronics|  North|Lucknow|2736.32|  Active|      2024-03-29|     Premium|
|           476|user_0476| 18|     Electronics|  North| Jaipur|2934.54| Pending|      2024-09-22|       Basic|
|           734|user_0734| 55|     Electronics|Central|Kolkata|1137.17| Pending|      2024-02-17|     Premium|
+--------------+---------+---+----------------+-------+-------+-------+--------+----------------+------------+
o

In [12]:
# ── 5c. Filter by Region ──────────────────────────────────────────────────
df_west = df_clean.filter(col("region") == "West")

print(f"West region transactions: {df_west.count()} rows")
df_west.show(5)


West region transactions: 197 rows


+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------------+
|transaction_id|  user_id|age|product_category|region|     city|  price|  status|transaction_date|subscription|
+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------------+
|           191|user_0191| 46|          Sports|  West|Ahmedabad|3446.18|Inactive|      2024-03-21|    Standard|
|           996|user_0996| 32|         Grocery|  West|Ahmedabad|3170.22|  Active|      2024-04-28|       Basic|
|           450|user_0450| 50|     Electronics|  West|  Kolkata| 678.07|  Active|      2024-09-13|       Basic|
|            60|user_0060| 38|         Grocery|  West|  Chennai|1694.13|  Active|      2024-07-13|    Standard|
|           888|user_0888| 51|     Electronics|  West|   Mumbai|1172.87| Pending|      2024-12-03|       Basic|
+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------

In [13]:
# ── 5d. Compound Filter: West + Premium ───────────────────────────────────
df_west_premium = df_clean.filter(
    (col("region") == "West") & (col("subscription") == "Premium")
)

print(f"West + Premium: {df_west_premium.count()} rows")
df_west_premium.show(5)


West + Premium: 62 rows


+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------------+
|transaction_id|  user_id|age|product_category|region|     city|  price|  status|transaction_date|subscription|
+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------------+
|           912|user_0912| 39|           Books|  West|  Chennai|3221.57|  Active|      2024-01-21|     Premium|
|           720|user_0720| 34|          Sports|  West|Hyderabad|2278.63|Inactive|      2024-03-03|     Premium|
|           804|user_0804| 44|          Sports|  West|   Jaipur| 698.59| Pending|      2024-08-07|     Premium|
|           350|user_0350| 28|           Books|  West|Bangalore| 2162.3|  Active|      2024-11-30|     Premium|
|           628|user_0628| 32|          Sports|  West|Ahmedabad|2386.57| Pending|      2024-09-29|     Premium|
+--------------+---------+---+----------------+------+---------+-------+--------+----------------+------

---
## Step 6 – Transform Data

Transformations modify the **structure** or **values** of the DataFrame — renaming columns, casting types, adding derived columns.

> **Immutability reminder:** every `.withColumn()` / `.withColumnRenamed()` / `.drop()` returns a **new** DataFrame. The original is never modified.


In [14]:
# ── 6a. Rename Columns ────────────────────────────────────────────────────
df_renamed = (
    df_clean
    .withColumnRenamed("product_category", "category")
    .withColumnRenamed("transaction_date", "txn_date")
)

print("Columns after renaming:", df_renamed.columns)
df_renamed.show(3)


Columns after renaming: ['transaction_id', 'user_id', 'age', 'category', 'region', 'city', 'price', 'status', 'txn_date', 'subscription']


+--------------+---------+---+--------+-------+---------+-------+--------+----------+------------+
|transaction_id|  user_id|age|category| region|     city|  price|  status|  txn_date|subscription|
+--------------+---------+---+--------+-------+---------+-------+--------+----------+------------+
|           463|user_0463| 29|  Sports|Central|  Lucknow| 756.31|  Active|2024-02-19|       Basic|
|           920|user_0920| 30|Clothing|  South|   Mumbai|1672.84|  Active|2024-08-28|       Basic|
|           683|user_0683| 28|   Books|   East|Bangalore|4781.51|Inactive|2024-11-12|       Basic|
+--------------+---------+---+--------+-------+---------+-------+--------+----------+------------+
only showing top 3 rows


In [15]:
# ── 6b. Change Data Types ─────────────────────────────────────────────────
# Cast price (DoubleType) to IntegerType for integer-level reporting
# Cast age to StringType just to demonstrate — then revert

df_transformed = (
    df_renamed
    .withColumn("price_int",   col("price").cast(IntegerType()))   # Double → Int
    .withColumn("price_round", spark_round(col("price"), 0))       # rounded Double
)

print("=== New columns (price_int, price_round) ===")
df_transformed.select("user_id", "price", "price_int", "price_round").show(5)


=== New columns (price_int, price_round) ===


+---------+-------+---------+-----------+
|  user_id|  price|price_int|price_round|
+---------+-------+---------+-----------+
|user_0463| 756.31|      756|      756.0|
|user_0920|1672.84|     1672|     1673.0|
|user_0683|4781.51|     4781|     4782.0|
|user_0191|3446.18|     3446|     3446.0|
|user_0861| 985.08|      985|      985.0|
+---------+-------+---------+-----------+
only showing top 5 rows


In [16]:
# ── 6c. Add a Derived Column ──────────────────────────────────────────────
# Label transactions as "High Value" (price >= 2000) or "Standard"
df_labeled = df_transformed.withColumn(
    "value_segment",
    when(col("price") >= 2000, "High Value").otherwise("Standard")
)

print("=== Value Segmentation ===")
df_labeled.select("user_id", "price", "value_segment").show(8)
df_labeled.groupBy("value_segment").count().show()


=== Value Segmentation ===


+---------+-------+-------------+
|  user_id|  price|value_segment|
+---------+-------+-------------+
|user_0463| 756.31|     Standard|
|user_0920|1672.84|     Standard|
|user_0683|4781.51|   High Value|
|user_0191|3446.18|   High Value|
|user_0861| 985.08|     Standard|
|user_0646|2674.36|   High Value|
|user_0195|1972.28|     Standard|
|user_0996|3170.22|   High Value|
+---------+-------+-------------+
only showing top 8 rows


+-------------+-----+
|value_segment|count|
+-------------+-----+
|     Standard|  403|
|   High Value|  597|
+-------------+-----+



---
## Step 7 – Aggregation

Aggregations compute **summary statistics** over the entire DataFrame (or grouped subsets).

> Calling multiple `.agg()` functions together is more efficient than chaining separate actions — Spark computes all statistics in a **single pass** over the data.


In [17]:
# ── 7a. Overall Statistics ────────────────────────────────────────────────
print("=== Overall Aggregation Statistics ===")
df_clean.agg(
    count("*")                          .alias("total_rows"),
    count("price")                      .alias("non_null_prices"),
    spark_round(avg("price"), 2)        .alias("avg_price"),
    spark_round(spark_min("price"), 2)  .alias("min_price"),
    spark_round(spark_max("price"), 2)  .alias("max_price"),
    spark_round(spark_sum("price"), 2)  .alias("total_revenue"),
    spark_round(stddev("price"), 2)     .alias("std_dev_price"),
    spark_round(avg("age"), 1)          .alias("avg_age"),
    spark_min("age")                    .alias("min_age"),
    spark_max("age")                    .alias("max_age"),
).show(truncate=False)


=== Overall Aggregation Statistics ===


+----------+---------------+---------+---------+---------+-------------+-------------+-------+-------+-------+
|total_rows|non_null_prices|avg_price|min_price|max_price|total_revenue|std_dev_price|avg_age|min_age|max_age|
+----------+---------------+---------+---------+---------+-------------+-------------+-------+-------+-------+
|1000      |1000           |2485.62  |0.0      |4999.54  |2485622.74   |1509.31      |39.0   |18     |60     |
+----------+---------------+---------+---------+---------+-------------+-------------+-------+-------+-------+



---
## Step 8 – Group Data with groupBy()

`groupBy()` is a **wide transformation** — it triggers a **shuffle** (data redistribution across partitions) so all rows with the same key land on the same executor before aggregation.


In [18]:
# ── 8a. Revenue by Product Category ──────────────────────────────────────
print("=== Revenue by Product Category ===")
df_clean.groupBy("product_category").agg(
    count("*")                         .alias("transaction_count"),
    spark_round(spark_sum("price"), 2) .alias("total_revenue"),
    spark_round(avg("price"), 2)       .alias("avg_price"),
    spark_round(spark_min("price"), 2) .alias("min_price"),
    spark_round(spark_max("price"), 2) .alias("max_price"),
).orderBy("total_revenue", ascending=False).show()


=== Revenue by Product Category ===


+----------------+-----------------+-------------+---------+---------+---------+
|product_category|transaction_count|total_revenue|avg_price|min_price|max_price|
+----------------+-----------------+-------------+---------+---------+---------+
|          Sports|              210|    556279.42|  2648.95|      0.0|  4982.59|
|           Books|              217|    512372.63|  2361.16|      0.0|  4997.98|
|        Clothing|              202|    495811.62|  2454.51|      0.0|  4994.12|
|         Grocery|              189|    476171.94|  2519.43|      0.0|   4990.0|
|     Electronics|              182|    444987.13|  2444.98|      0.0|  4999.54|
+----------------+-----------------+-------------+---------+---------+---------+



In [19]:
# ── 8b. Transaction Count by Region ───────────────────────────────────────
print("=== Transactions by Region ===")
df_clean.groupBy("region").agg(
    count("*")                         .alias("transactions"),
    spark_round(spark_sum("price"), 2) .alias("total_revenue"),
    spark_round(avg("price"), 2)       .alias("avg_order_value"),
).orderBy("total_revenue", ascending=False).show()


=== Transactions by Region ===


+-------+------------+-------------+---------------+
| region|transactions|total_revenue|avg_order_value|
+-------+------------+-------------+---------------+
|Central|         220|    536596.24|        2439.07|
|  South|         205|    510854.55|        2491.97|
|  North|         189|    495564.72|        2622.04|
|   East|         189|    476108.79|        2519.09|
|   West|         197|    466498.44|        2368.01|
+-------+------------+-------------+---------------+



In [20]:
# ── 8c. City-level Count — Only Cities with > 50 Transactions ────────────
print("=== Cities with > 50 Transactions ===")
df_clean.groupBy("city").count()     .filter("count > 50")     .orderBy("count", ascending=False)     .show()


=== Cities with > 50 Transactions ===


+---------+-----+
|     city|count|
+---------+-----+
|     Pune|  118|
|Ahmedabad|  109|
|  Chennai|  107|
|Bangalore|  106|
|   Jaipur|  105|
|    Delhi|   96|
|   Mumbai|   94|
|  Kolkata|   93|
|Hyderabad|   90|
|  Lucknow|   82|
+---------+-----+



In [21]:
# ── 8d. Revenue by Subscription Tier ─────────────────────────────────────
print("=== Revenue by Subscription Tier ===")
df_clean.groupBy("subscription").agg(
    count("*")                         .alias("users"),
    spark_round(spark_sum("price"), 2) .alias("total_revenue"),
    spark_round(avg("price"), 2)       .alias("avg_price"),
).orderBy("total_revenue", ascending=False).show()


=== Revenue by Subscription Tier ===


+------------+-----+-------------+---------+
|subscription|users|total_revenue|avg_price|
+------------+-----+-------------+---------+
|     Premium|  346|    895668.73|  2588.64|
|    Standard|  320|    809410.36|  2529.41|
|       Basic|  334|    780543.65|  2336.96|
+------------+-----+-------------+---------+



In [22]:
# ── 8e. Region × Category Cross-Analysis ─────────────────────────────────
print("=== Region × Category Revenue Breakdown ===")
df_clean.groupBy("region", "product_category").agg(
    count("*")                         .alias("txn_count"),
    spark_round(spark_sum("price"), 2) .alias("revenue"),
).orderBy("region", "revenue", ascending=[True, False]).show(25)


=== Region × Category Revenue Breakdown ===


+-------+----------------+---------+---------+
| region|product_category|txn_count|  revenue|
+-------+----------------+---------+---------+
|Central|        Clothing|       48|134914.29|
|Central|     Electronics|       41|113724.12|
|Central|          Sports|       43|100801.05|
|Central|           Books|       47| 99895.55|
|Central|         Grocery|       41| 87261.23|
|   East|          Sports|       47|146589.36|
|   East|           Books|       40|100146.22|
|   East|     Electronics|       40| 90992.56|
|   East|        Clothing|       33| 69874.81|
|   East|         Grocery|       29| 68505.84|
|  North|     Electronics|       41|111070.47|
|  North|          Sports|       41|106616.51|
|  North|         Grocery|       35|104004.27|
|  North|        Clothing|       36| 89276.68|
|  North|           Books|       36| 84596.79|
|  South|           Books|       50|125699.92|
|  South|        Clothing|       46|113868.49|
|  South|         Grocery|       42|104733.99|
|  South|    

---
## Step 9 – Wide Transformations, Shuffle & the Execution Plan

### Narrow vs Wide Transformations

| Type | Examples | Data Movement |
|------|----------|---------------|
| **Narrow** | `filter`, `select`, `withColumn`, `map` | None — each partition processed independently |
| **Wide** | `groupBy`, `join`, `distinct`, `repartition` | Shuffle — data crosses partition/executor boundaries |

### What Happens During a Shuffle?
1. **Map phase:** each executor hashes the groupBy key for every row and writes records to local shuffle files.
2. **Transfer:** records are sent across the network to the executor responsible for that key hash.
3. **Reduce phase:** the target executor receives all rows for its assigned keys and computes the aggregation.

Shuffles are expensive (network I/O, serialization, disk spill). Minimize them by:
- Filtering data **before** groupBy (reduce rows going into the shuffle)
- Using `repartition("key_col")` before multiple groupBys on the same key
- Keeping `spark.sql.shuffle.partitions` tuned to your data size (default 200 is too high for small data)


In [23]:
# ── 9a. View the Execution Plan (DAG) ────────────────────────────────────
# "Exchange hashpartitioning" in the plan = the shuffle step
print("=== Execution Plan for groupBy ===")
df_clean.groupBy("region").agg(spark_sum("price")).explain()


=== Execution Plan for groupBy ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#434], functions=[sum(price#375)])
   +- Exchange hashpartitioning(region#434, 4), ENSURE_REQUIREMENTS, [plan_id=2181]
      +- HashAggregate(keys=[region#434], functions=[partial_sum(price#375)])
         +- HashAggregate(keys=[city#5, price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3], functions=[])
            +- Exchange hashpartitioning(city#5, price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3, 4), ENSURE_REQUIREMENTS, [plan_id=2177]
               +- HashAggregate(keys=[city#5, knownfloatingpointnormalized(normalizenanandzero(price#6)) AS price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3], functions=[])
                  +- Filter (atleastnnonnulls(1, age#

In [24]:
# ── 9b. Narrow Transformation Plan (no shuffle) ───────────────────────────
print("=== Execution Plan for filter (narrow — no Exchange) ===")
df_clean.filter(col("region") == "West").explain()


=== Execution Plan for filter (narrow — no Exchange) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#5, price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3], functions=[])
   +- Exchange hashpartitioning(city#5, price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3, 4), ENSURE_REQUIREMENTS, [plan_id=2196]
      +- HashAggregate(keys=[city#5, knownfloatingpointnormalized(normalizenanandzero(price#6)) AS price#6, subscription#9, age#2, user_id#1, transaction_id#0, status#7, region#4, transaction_date#8, product_category#3], functions=[])
         +- Filter ((atleastnnonnulls(1, age#2) AND atleastnnonnulls(1, CASE WHEN ((age#2 >= 18) AND (age#2 <= 100)) THEN age#2 END)) AND (trim(region#4, None) = West))
            +- FileScan csv [transaction_id#0,user_id#1,age#2,product_category#3,region#4,city#5,price#6,status#7,t

In [25]:
# ── 9c. Repartition to reduce shuffle cost for repeated groupBys ──────────
# If you're doing multiple groupBy("region") operations, repartition first
df_by_region = df_clean.repartition(4, "region")
print(f"Partitions after repartition: {df_by_region.rdd.getNumPartitions()}")


Partitions after repartition: 4


---
## Step 10 – Final Pipeline

This brings together all steps into a single, production-style pipeline:
1. Load raw CSV
2. Deduplicate
3. Handle nulls
4. Clean schema
5. Aggregate revenue per store/category
6. Write results


In [26]:
# ══════════════════════════════════════════════════════════════════════════
# FINAL END-TO-END PIPELINE
# ══════════════════════════════════════════════════════════════════════════

import os
os.makedirs("output", exist_ok=True)

print("=" * 60)
print("  WEEK 5 — SPARK DATA PIPELINE")
print("=" * 60)

# ── STEP 1: Load ──────────────────────────────────────────────────────────
print("\n[1/5] Loading raw data...")
schema = StructType([
    StructField("transaction_id",   IntegerType(),  True),
    StructField("user_id",          StringType(),   True),
    StructField("age",              IntegerType(),  True),
    StructField("product_category", StringType(),   True),
    StructField("region",           StringType(),   True),
    StructField("city",             StringType(),   True),
    StructField("price",            DoubleType(),   True),
    StructField("status",           StringType(),   True),
    StructField("transaction_date", StringType(),   True),
    StructField("subscription",     StringType(),   True),
])
df_pipeline = spark.read.option("header","true").option("nullValue","").schema(schema).csv("data/ecommerce_data.csv")
raw_count = df_pipeline.count()
print(f"    Rows loaded: {raw_count}")

# ── STEP 2: Deduplicate ───────────────────────────────────────────────────
print("\n[2/5] Removing duplicates...")
df_pipeline = df_pipeline.dropDuplicates()
after_dedup = df_pipeline.count()
print(f"    Duplicates removed: {raw_count - after_dedup}")
print(f"    Rows remaining    : {after_dedup}")

# ── STEP 3: Handle Nulls ─────────────────────────────────────────────────
print("\n[3/5] Handling null values...")
df_pipeline = df_pipeline.na.fill({"price": 0.0, "status": "Unknown"})
df_pipeline = df_pipeline.na.drop(subset=["age", "user_id"])
print(f"    Rows after null handling: {df_pipeline.count()}")

# ── STEP 4: Clean Schema & Add Features ──────────────────────────────────
print("\n[4/5] Cleaning schema and transforming columns...")
df_pipeline = (
    df_pipeline
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("region",           trim(col("region")))
    .withColumn("subscription",     trim(col("subscription")))
    .withColumn("transaction_date", to_date(col("transaction_date"), "yyyy-MM-dd"))
    .withColumn("price",            spark_round(col("price"), 2))
    .withColumn("age",              when((col("age") >= 18) & (col("age") <= 100), col("age")).otherwise(None))
    .withColumn("value_segment",    when(col("price") >= 2000, "High Value").otherwise("Standard"))
    .na.drop(subset=["age"])
)
final_count = df_pipeline.count()
print(f"    Final clean rows: {final_count}")

# ── STEP 5: Aggregate ─────────────────────────────────────────────────────
print("\n[5/5] Running aggregations...")

# Revenue by category
df_cat_revenue = (
    df_pipeline
    .groupBy("product_category")
    .agg(
        count("*")                         .alias("transactions"),
        spark_round(spark_sum("price"), 2) .alias("total_revenue"),
        spark_round(avg("price"), 2)       .alias("avg_order_value"),
    )
    .orderBy("total_revenue", ascending=False)
)

# Revenue by region
df_region_revenue = (
    df_pipeline
    .groupBy("region")
    .agg(
        count("*")                         .alias("transactions"),
        spark_round(spark_sum("price"), 2) .alias("total_revenue"),
        spark_round(avg("price"), 2)       .alias("avg_order_value"),
    )
    .orderBy("total_revenue", ascending=False)
)

# Top cities
df_top_cities = (
    df_pipeline
    .groupBy("city")
    .agg(
        count("*")                         .alias("transactions"),
        spark_round(spark_sum("price"), 2) .alias("total_revenue"),
    )
    .orderBy("total_revenue", ascending=False)
    .limit(5)
)

print("\n📊 Revenue by Product Category:")
df_cat_revenue.show()

print("📊 Revenue by Region:")
df_region_revenue.show()

print("📊 Top 5 Cities by Revenue:")
df_top_cities.show()

# ── Write Output ──────────────────────────────────────────────────────────
df_cat_revenue.coalesce(1).write.mode("overwrite").option("header","true").csv("output/revenue_by_category")
df_region_revenue.coalesce(1).write.mode("overwrite").option("header","true").csv("output/revenue_by_region")

print("\n" + "=" * 60)
print("  ✅ PIPELINE COMPLETE")
print("=" * 60)
print(f"  Raw rows loaded     : {raw_count}")
print(f"  Duplicates removed  : {raw_count - after_dedup}")
print(f"  Final clean rows    : {final_count}")
print(f"  Output saved to     : ./output/")
print("=" * 60)


  WEEK 5 — SPARK DATA PIPELINE

[1/5] Loading raw data...
    Rows loaded: 1030

[2/5] Removing duplicates...


    Duplicates removed: 30
    Rows remaining    : 1000

[3/5] Handling null values...


    Rows after null handling: 1000

[4/5] Cleaning schema and transforming columns...


    Final clean rows: 1000

[5/5] Running aggregations...

📊 Revenue by Product Category:


+----------------+------------+-------------+---------------+
|product_category|transactions|total_revenue|avg_order_value|
+----------------+------------+-------------+---------------+
|          Sports|         210|    556279.42|        2648.95|
|           Books|         217|    512372.63|        2361.16|
|        Clothing|         202|    495811.62|        2454.51|
|         Grocery|         189|    476171.94|        2519.43|
|     Electronics|         182|    444987.13|        2444.98|
+----------------+------------+-------------+---------------+

📊 Revenue by Region:


+-------+------------+-------------+---------------+
| region|transactions|total_revenue|avg_order_value|
+-------+------------+-------------+---------------+
|Central|         220|    536596.24|        2439.07|
|  South|         205|    510854.55|        2491.97|
|  North|         189|    495564.72|        2622.04|
|   East|         189|    476108.79|        2519.09|
|   West|         197|    466498.44|        2368.01|
+-------+------------+-------------+---------------+

📊 Top 5 Cities by Revenue:


+---------+------------+-------------+
|     city|transactions|total_revenue|
+---------+------------+-------------+
|     Pune|         118|    310575.03|
|Bangalore|         106|    292098.11|
|Ahmedabad|         109|    259628.78|
|   Jaipur|         105|    256662.55|
|Hyderabad|          90|    250760.37|
+---------+------------+-------------+




  ✅ PIPELINE COMPLETE
  Raw rows loaded     : 1030
  Duplicates removed  : 30
  Final clean rows    : 1000
  Output saved to     : ./output/


---
## Summary & Key Takeaways

| Step | What We Did | Spark Concept |
|------|------------|---------------|
| 1 | Understood MapReduce vs Spark | In-memory processing, DAG, lazy eval |
| 2 | Created SparkSession | Entry point for all Spark operations |
| 3 | Loaded CSV with explicit schema | `spark.read.schema().csv()` |
| 4 | Removed duplicates, filled/dropped nulls | `.dropDuplicates()`, `.na.fill()`, `.na.drop()` |
| 5 | Filtered by age, category, region | Narrow transformations, Catalyst pushdown |
| 6 | Renamed columns, cast types, added features | `.withColumnRenamed()`, `.cast()`, `.withColumn()` |
| 7 | Computed overall statistics | `.agg()` with min/max/avg/sum/stddev |
| 8 | Grouped data and computed revenue | Wide transformation, shuffle |
| 9 | Inspected execution plans | `.explain()`, narrow vs wide, shuffle cost |
| 10 | Built complete end-to-end pipeline | Production-style chained DataFrame operations |

### Pipeline Design Principles Applied
- **Deduplicate first** — avoids inflating all downstream counts and sums
- **Fill nulls before aggregating** — null prices silently skew avg() and sum()
- **Filter before groupBy** — reduces data shuffled across the network
- **Explicit schema** — faster load, prevents type inference errors on messy dates
- **Immutability** — each transformation creates a new DataFrame; originals are never mutated


In [27]:
# Clean up — stop the Spark session
spark.stop()
print("Spark session stopped.")


Spark session stopped.
